In [ ]:
# Run from the repository root, or open this notebook anywhere inside it.
from pathlib import Path
import os
_repo_candidates = [Path.cwd(), *Path.cwd().parents]
_project_root = next((p for p in _repo_candidates if (p / "code/data_process").is_dir() and (p / "README.md").is_file()), None)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.chdir(_project_root)


# Read all waves -> filter data 

In [ ]:
import pandas as pd
import os

# Define the base path
base_path = "./data-all-waves"

# Create dictionaries to store original and filtered dataframes
dfs = {}
filtered_dfs = {}

# Define the years
years = ['10', '13', '15', '17', '19', '21']

# Read and filter each file
for year in years:
    file_name = f"epcg{year}.sas7bdat"
    file_path = os.path.join(base_path, file_name)
    
    print(f"\nProcessing year 20{year}:")
    
    try:
        # Read with latin1 encoding
        dfs[f'df_{year}'] = pd.read_sas(file_path, encoding='latin1')
        
        # Filter the data
        filtered_dfs[f'df_{year}_filtered'] = dfs[f'df_{year}'][
            (dfs[f'df_{year}']['LFSTAT'] == '1') & 
            (dfs[f'df_{year}']['EMUS'] == 'Y') & 
            ((dfs[f'df_{year}']['DGRDG'] == '1') | (dfs[f'df_{year}']['DGRDG'] == '2'))
        ]
        
        # Print information about the filtering
        print(f"Original rows: {len(dfs[f'df_{year}'])}")
        print(f"Filtered rows: {len(filtered_dfs[f'df_{year}_filtered'])}")
        print(f"Percentage kept: {(len(filtered_dfs[f'df_{year}_filtered'])/len(dfs[f'df_{year}'])*100):.2f}%")
        
    except Exception as e:
        print(f"Error processing data for 20{year}: {str(e)}")

# Select variables (and make the names consistent)

In [ ]:
# Define variables that are consistent across all waves
common_vars = ['REFYR','REFID', 'BIRYR','COHORT', 'AGE', 'CTZN', 'GENDER', 'RACEM', 
               'BTHUS','BTHRGN','BTHST','EDDAD','EDMOM','HSST','HSYR',              
               'HRSWK', 'SALARY', 
               'MGRNAT', 'MGROTH', 'MGRSOC', 'BAYR', 'DGRYR', 'D2YR', 'D3YR', 'D4YR', 'D5YR',
               'BAMN', 'HDMN', 'D2MN', 'D3MN', 'D4MN', 'D5MN', 'DGRDG', 'D2DG', 'D3DG', 'D4DG', 'D5DG',
               'NBAMEMG', 'NBAMENG', 'NBASEMG', 'NBANENG', 'NDGMEMG', 'NDGMENG',
               'ND2MEMG', 'ND2MENG', 'ND2SEMG', 'ND2NENG', 'ND3MEMG', 'ND3MENG',
               'EMSIZE', 'EMSECSM', 'EMSECDT', 'BSDGN', 'BADGRUS', 'LFSTAT', 'EMUS',
               'STRTMN', 'STRTYR', 'LWMN', 'LWYR', 'TCDGCMP', 'NRREA']

# Define variable mappings for those that change names
var_mappings = {
    'pre_2019': {
        'NBAMED': 'NBAMED',
        'NBANED': 'NBANED',
        'NDGRMED': 'NDGRMED',
        'ND2MED': 'ND2MED',
        'ND2NED': 'ND2NED',
        'ND3MED': 'ND3MED'
    },
    'post_2019': {
        'NBAMED': 'N2BAMED',
        'NBANED': 'N2BANED',
        'NDGRMED': 'N2DGRMED',
        'ND2MED': 'N2D2MED',
        'ND2NED': 'N2D2NED',
        'ND3MED': 'N2D3MED'
    }
}

# Create dictionary to store selected dataframes
selected_dfs = {}

# Process each year
for year in years:
    print(f"\nProcessing year 20{year}:")
    
    try:
        # Determine which variable names to use based on year
        if int(year) < 19:  # Before 2019
            year_specific_vars = var_mappings['pre_2019']
        else:  # 2019 and after
            year_specific_vars = var_mappings['post_2019']
        
        # Combine common variables with year-specific variables
        all_vars = common_vars + list(year_specific_vars.values())
        
        # Select only variables that exist in the dataset
        existing_vars = [var for var in all_vars if var in filtered_dfs[f'df_{year}_filtered'].columns]
        
        # Select variables
        selected_dfs[f'df_{year}_selected'] = filtered_dfs[f'df_{year}_filtered'][existing_vars]
        
        # Print information
        print(f"Original columns: {len(filtered_dfs[f'df_{year}_filtered'].columns)}")
        print(f"Selected columns: {len(existing_vars)}")
        print(f"Rows: {len(filtered_dfs[f'df_{year}_filtered'])}")
        
        # Print any variables that were in our list but not found in the dataset
        missing_vars = set(all_vars) - set(existing_vars)
        if missing_vars:
            print(f"Warning: The following variables were not found in 20{year} data:")
            print(missing_vars)
            
    except Exception as e:
        print(f"Error processing data for 20{year}: {str(e)}")
## Verify the selected variables for one year (e.g., 2010)
print("\nSelected variables for 2010:")
print(selected_dfs['df_10_selected'].columns.tolist())

print("\nSelected variables for 2017:")
print(selected_dfs['df_17_selected'].columns.tolist())

print("\nSelected variables for 2019:")
print(selected_dfs['df_19_selected'].columns.tolist())

In [ ]:
selected_dfs

# Merge all waves into one dataframe

In [ ]:
 # First, let's standardize the variable names (converting post-2019 names to pre-2019 format)
def standardize_names(df, year):
    if int(year) >= 19:  # For 2019 and 2021
        rename_dict = {
            'N2BAMED': 'NBAMED',
            'N2BANED': 'NBANED',
            'N2DGRMED': 'NDGRMED',
            'N2D2MED': 'ND2MED',
            'N2D2NED': 'ND2NED',
            'N2D3MED': 'ND3MED'
        }
        return df.rename(columns=rename_dict)
    return df

# Create a list to store standardized dataframes
standardized_dfs = []

# Process each year
for year in years:
    # Get the selected dataframe for this year
    df = selected_dfs[f'df_{year}_selected'].copy()
    
    # Add a year identifier column
    df['SURVEY_YEAR'] = f'20{year}'
    
    # Standardize variable names
    df = standardize_names(df, year)
    
    # Append to list
    standardized_dfs.append(df)
    
    print(f"Processed 20{year}: {df.shape[0]} rows, {df.shape[1]} columns")

# Merge all dataframes
merged_df = pd.concat(standardized_dfs, axis=0, ignore_index=True)

# Print information about the merged dataset
print("\nMerged dataset information:")
print(f"Total rows: {merged_df.shape[0]}")
print(f"Total columns: {merged_df.shape[1]}")
print("\nNumber of observations by survey year:")
print(merged_df['SURVEY_YEAR'].value_counts().sort_index())

# Verify that all variable names are consistent
print("\nAll columns in merged dataset:")
print(sorted(merged_df.columns.tolist()))

# Optional: Check for any missing values in key variables
print("\nMissing values in key variables:")
print(merged_df[['SURVEY_YEAR', 'REFYR', 'AGE', 'SALARY']].isnull().sum())

# Convert REFYR from float strings ('2010.0') to integers (2010)
merged_df['REFYR'] = merged_df['REFYR'].astype(float).astype(int)

# Verify the conversion
print("\nREFYR values after conversion:")
print(merged_df['REFYR'].unique())


merged_df = merged_df.drop('SURVEY_YEAR', axis=1)
print("\nSURVEY_YEAR removed as it matches REFYR")

In [ ]:
merged_df

### Generate BIRYR for years 2010, 2013 and 2015

In [ ]:
# Create mask for years 2010, 2013, and 2015
mask = merged_df['REFYR'].isin([2010, 2013, 2015])

# Calculate BIRYR only for those years
merged_df.loc[mask, 'BIRYR'] = merged_df.loc[mask, 'REFYR'] - merged_df.loc[mask, 'AGE'] - 1

# Verify the calculation
print("\nVerification of BIRYR calculation:")
for year in [2010, 2013, 2015, 2017, 2019, 2021]:
    print(f"\nYear {year}:")
    print(merged_df[merged_df['REFYR']==year][['REFYR', 'AGE', 'BIRYR']].head())
    print(f"Number of non-null BIRYR values for {year}: {merged_df[(merged_df['REFYR']==year) & (merged_df['BIRYR'].notna())].shape[0]}")

### Remind: be careful about matching using BIRYR -> may have 1 year error

In [ ]:
# Create a function to analyze BIRYR patterns
def analyze_biryr_pattern(df, year):
    # Create mask for the specific year
    year_mask = df['REFYR'] == year
    # Get subset of data for this year where BIRYR is not null
    year_data = df[year_mask & df['BIRYR'].notna()]
    
    # Count cases where BIRYR = REFYR-AGE
    pattern1 = (year_data['REFYR'] - year_data['AGE'] == year_data['BIRYR']).sum()
    
    # Count cases where BIRYR = REFYR-AGE-1
    pattern2 = (year_data['REFYR'] - year_data['AGE'] - 1 == year_data['BIRYR']).sum()
    
    # Total cases with non-null BIRYR
    total = len(year_data)
    
    # Cases that don't follow either pattern
    other = total - pattern1 - pattern2
    
    print(f"\nYear {year}:")
    print(f"Total observations with non-null BIRYR: {total}")
    print(f"BIRYR = REFYR-AGE: {pattern1} ({pattern1/total*100:.2f}%)")
    print(f"BIRYR = REFYR-AGE-1: {pattern2} ({pattern2/total*100:.2f}%)")
    print(f"Other patterns: {other} ({other/total*100:.2f}%)")
    
    return total, pattern1, pattern2, other

# Analyze patterns for each year
for year in [2017, 2019, 2021]:
    analyze_biryr_pattern(merged_df, year)

# Export to .csv

In [ ]:
# Now save to CSV
output_file = os.path.join(base_path, "update_merged_nscg_2010_to_2021.csv")
merged_df.to_csv(output_file, index=False)

print(f"\nDataset saved to: {output_file}")
print(f"Number of rows: {merged_df.shape[0]}")
print(f"Number of columns: {merged_df.shape[1]}")